In [4]:
import pandas as pd
from io import StringIO
from io import BytesIO
from googleapiclient.http import MediaIoBaseDownload
import re
import os
from IPython.display import clear_output

In [5]:
# this version of API V3 works for both shared folders and shared drives
# shared drives search and read is faster than shared folders - recommending all files are on shared drives
from google.oauth2 import service_account
from googleapiclient.discovery import build

SCOPES = ['https://www.googleapis.com/auth/drive.readonly']
SERVICE_ACCOUNT_FILE = 'biopilot-458819-531753a47910.json'  # or your credentials file

creds = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE, scopes=SCOPES)

service = build('drive', 'v3', credentials=creds)

drive_id = '0ABY9m1yN-I4zUk9PVA' # this is the shared drive id for both 10k metadata and reports

In [6]:
# find all files in folder_id (including subfolders) starting with prefix. 
# if looking in a shared drive - need the drive_id (root)
def find_all_files(folder_id, drive_id, path, prefix):
    rows = []

    query = f"'{folder_id}' in parents and trashed = false"

    page_token = None
    while True:
        if drive_id:
            response = service.files().list(
                q=query,
                spaces='drive',
                corpora='drive' if drive_id else 'default',
                driveId=drive_id,
                includeItemsFromAllDrives=True,
                supportsAllDrives=True,
                fields="nextPageToken, files(id, name, mimeType, parents)",
                pageToken=page_token
            ).execute()
        else:
            response = service.files().list(
                q=query,
                includeItemsFromAllDrives=True,
                supportsAllDrives=True,
                fields="nextPageToken, files(id, name, mimeType, parents)",
                pageToken=page_token
            ).execute()
            
        for file in response.get('files', []):
            if file['mimeType'] == 'application/vnd.google-apps.folder':
                # Recurse into subfolder
                rows.extend(find_all_files(file['id'], drive_id, path+file['name']+'/', prefix))
            elif file['name'].startswith(prefix):
                rows.append({'id': file['id'],'date': path[:-1]})
                clear_output(wait=True)
                print(f"Adding file number {len(rows)}: {path+file['name']} ")

        page_token = response.get('nextPageToken', None)
        if not page_token:
            break

    return rows

In [32]:
# get all stationary walk data report files - from folder 
folder_id = '10ZJNkpxSnNUNe51Jrq6_Kgr7MxsQfZ3X' # RnD\Algorithm\Weizmann-GS&MS\Reports\2025\WZ-SWe\04\25.04.25--2100_ds--with_batch_summary4SWe
sw_file_table = find_all_files(folder_id, '', '', 'swe_report')
df = pd.DataFrame(sw_file_table)
df.to_csv('sw_file_table.csv', index=False)

Adding file number 1: session_id_29a74a08-932a-4bc4-b1fa-794be3789bdc/swe_report.xlsx 


In [31]:
# get all meta files - from folder "10k export metadata"
folder_id = '1-7MV8hYagSW-veGcJJHJBhVZQJTAE4cz' # "10k export metadata"
meta_file_table = find_all_files(folder_id, drive_id, '', '')
df = pd.DataFrame(meta_file_table)
df.to_csv('meta_file_table.csv', index=False)

Adding file number 10: 2023/11/01/0db16583-dc35-41ec-8088-be6fd8524d01.csv 


2246

In [ ]:
# get all data report files - from folder "WIS p10k reports"
folder_id = '1t19J84Bm0Pzl3gTjOazNYjkwDag_uUrZ' # "WIS p10k reports"
data_file_table = find_all_files(folder_id, drive_id, '', 'summary')
df = pd.DataFrame(data_file_table)
df.to_csv('data_file_table.csv', index=False)

In [7]:
# read and merge all single rows from metadata files to meta table

def read_meta_file(file_id):
    request = service.files().get_media(fileId=file_id)
    fh = BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    fh.seek(0)
    df = pd.read_csv(fh)
    return df.iloc[[0]]  # ensure it's a DataFrame with one row
    
def merge_metadata(file_list):
    rows = []
    for file in file_list:
        file_id = file['id']
        date = file['date']
        try:
            row_df = read_meta_file(file_id)
            row_df['meta_file_date'] = date
            row_df['meta_file_id'] = file_id
            rows.append(row_df)
        except Exception as e:
            print(f"Failed to process {name}: {e}")
        print(len(rows))
    return pd.concat(rows, ignore_index=True)

In [8]:
# read all stationary walk swe_report - SWe_statistics sheet
# and merge to one table

# read Stationary Walk swe reports
def read_swe_report(file_id):

    request = service.files().get_media(fileId=file_id)
    fh = BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    fh.seek(0)

    # Load only the target sheet
    df = pd.read_excel(fh, sheet_name='SWe-statistics', index_col=0)
    
    # Flatten into a single-row dict with prefixed keys
    flat_dict = {
        f'SW_{row}_{col}': df.at[row, col]
        for row in df.index
        for col in df.columns
    }
    
    # Convert to single-row DataFrame
    row_df = pd.DataFrame([flat_dict])
    
    return row_df
    
def merge_sw_data(file_list):
    rows = []
    for file in file_list:
        file_id = file['id']
        try:
            row_df = read_swe_report(file_id)
            row_df['swe_file_id'] = file_id
            row_df['test_id'] = re.search(r'session_id_([a-f0-9\-]+)', file['date']).group(1)
            rows.append(row_df)
        except Exception as e:
            print(f"Failed to process {file['date']}: {e}")
        clear_output(wait=True)
        print(f"{len(rows)} / {len(file_list)} files processed")
    return pd.concat(rows, ignore_index=True)

In [57]:
# read all stationary walk swe_report - SWe_mresults sheet
# and merge to one table

# read Stationary Walk swe reports
def read_sw_mresults(file_id):

    request = service.files().get_media(fileId=file_id)
    fh = BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    fh.seek(0)

    # Load only the target sheet
    df_raw = pd.read_excel(fh, sheet_name='SWe-mresults', header=None, skiprows=1)
    
    # Use the first and second columns directly
    df_raw.columns = ['feature', 'value']
    
    # Convert to dict, then to single-row DataFrame
    row = pd.DataFrame([df_raw.set_index('feature')['value'].astype(float).to_dict()])
    
    # Function to rename columns by moving stat suffix to end
    def reorder_stat_suffix(col):
        stat_keywords = ['mean', 'std', 'median', 'cv']
        for stat in stat_keywords:
            pattern = rf'(_{stat})(?=(_|$))'
            match = re.search(pattern, col, flags=re.IGNORECASE)
            if match:
                prefix = col[:match.start()]
                suffix = col[match.end():]
                return f"{prefix}{suffix}_{stat.lower()}"
        return col
    
    # Apply renaming
    row.columns = [reorder_stat_suffix(c) for c in row.columns]
    
    # Final DataFrame
    return row.head(1)

  
def merge_sw_mresults(file_list):
    rows = []
    for file in file_list:
        file_id = file['id']
        try:
            row_df = read_sw_mresults(file_id)
            row_df['swe_file_id'] = file_id
            row_df['test_id'] = re.search(r'session_id_([a-f0-9\-]+)', file['date']).group(1)
            rows.append(row_df)
        except Exception as e:
            print(f"Failed to process {file['date']}: {e}")
        clear_output(wait=True)
        print(f"{len(rows)} / {len(file_list)} files processed")
    return pd.concat(rows, ignore_index=True)

In [ ]:
# read and merge all summary reports into one table

def read_summary_report(file_id):

    request = service.files().get_media(fileId=file_id)
    fh = BytesIO()
    downloader = MediaIoBaseDownload(fh, request)
    done = False
    while not done:
        _, done = downloader.next_chunk()
    fh.seek(0)

    # Step 1: Load Excel with 3-row header
    df = pd.read_excel(fh, header=[0, 1, 2])
    
    # Step 2: Drop empty/unnamed stat columns
    df = df.loc[:, ~df.columns.get_level_values(2).str.contains('^Unnamed', na=False)]
    
    # Step 3: Assume first column is metric name
    df.index = df.iloc[:, 0]
    df = df.iloc[:, 1:]
    
    # Step 4: Prefix map including combined top headers
    prefix_map = {
        'Gait and Dynamic Balance (TM 3kmh)_Stats': 'TM3',
        'Gait and Dynamic Balance (TM Self Pace)_Stats': 'TMS',
        'Gait and Dynamic Balance (Stationary Walking)_Stats': 'SW',
        'Static Balance_Romberg Eyes open (EO)': 'EO',
        'Static Balance_Romberg Eyes closed (EC)': 'EC',
        'Static Balance_Romberg ratio (EC/EO)': 'EC2EO',
        'Static Balance_Strength (Sit to Stand)': 'S2S',
    }
    
    # Step 5: Generate new column names
    flattened = {}
    
    for metric in df.index:
        for col in df.columns:
            level0, level1, stat = col
            full_prefix_key = f"{level0.strip()}_{level1.strip()}"
            prefix = prefix_map.get(full_prefix_key, full_prefix_key.replace(' ', '_'))
            if prefix:  # only add if prefix is not blank
                col_name = f"{prefix}_{metric}_{stat.strip()}"
            else:
                col_name = f"{metric}_{stat.strip()}"
            flattened[col_name] = df.loc[metric, col]
    
    # Step 6: Final DataFrame
    row_df = pd.DataFrame([flattened])
    return row_df

# merge each downloaded row with the date (from the path) information
def merge_summary_report(file_list):
    rows = []
    for file in file_list:
        file_id = file['id']
        # there was an extra folder for session name after the date folders
        file_date, file_session = file['date'].rsplit('/', 1) 
        try:
            row_df = read_summary_report(file_id)
            row_df['data_file_date'] = file_date
            row_df['data_file_id'] = file_id
            row_df['test_id'] = file_session
            rows.append(row_df)
        except Exception as e:
            print(f"Failed to process {name}: {e}")
        clear_output(wait=True)
        print(f"{len(rows)} / {len(file_list)} files processed")
    return pd.concat(rows, ignore_index=True)

In [56]:
# download and merge all stat walk files
sw_file_table = pd.read_csv('sw_file_table.csv').to_dict(orient='records')
sw_mresults_table = merge_sw_mresults(sw_file_table)
sw_mresults_table.to_csv('sw_mresults_table.csv', index=False)

1975 / 2000 files processed


In [ ]:
# download and merge all stat walk files
sw_table = merge_sw_data(sw_file_table)
sw_table.to_csv('sw_table.csv', index=False)

In [ ]:
# download and merge all meta files
meta_table = merge_metadata(meta_file_table)
meta_table.to_csv('meta_table.csv', index=False)

In [ ]:
# download and merge all summary files
# delete all blank columns after merge 
data_table = merge_summary_report(data_file_table)

initial_count = len(data_table)
data_table = data_table.dropna(axis=1, how='all')
dropped_count = initial_count - len(data_table)

print(f"Dropped {dropped_count} rows out of {initial_count} total ({dropped_count / initial_count:.1%}).")

data_table.to_csv('data_table.csv', index=False)

In [3]:
# merge meta table with data table
meta_table = pd.read_csv('meta_table.csv')
data_table = pd.read_csv('data_table.csv')

data_table = data_table.rename(columns={'test_id': 'test_id_data'})
meta_table = meta_table.rename(columns={'test_id': 'test_id_meta'})

merged_all = pd.merge(
    data_table, meta_table,
    left_on='test_id_data', right_on='test_id_meta',
    how='outer',
    indicator=True
)

# Rename categorical labels properly
merged_all['_merge'] = merged_all['_merge'].cat.rename_categories({
    'left_only': 'Unmatched rows in data_table',
    'right_only': 'Unmatched rows in meta_table',
    'both': 'Matched rows'  # optional, keeps 'both' unchanged
})

# map merging result
counts = merged_all['_merge'].value_counts().to_dict()
print(f"results of merging {len(merged_all)} entries:")
print(counts)

# Reorder columns
first_cols = ['_merge', 'test_id_data', 'test_id_meta', 'meta_file_id', 'data_file_id', 'meta_file_date', 'data_file_date']
meta_cols = [col for col in meta_table.columns if col not in first_cols]
other_cols = [col for col in merged_all.columns if col not in first_cols + meta_cols]

# Final column order
final_cols = first_cols + meta_cols + other_cols
merged_all = merged_all[final_cols]

# Keep records of unmatched rows
unmatched = merged_all[merged_all['_merge'] != 'Matched rows'].copy()

# Save
unmatched.to_csv('unmatched_records_data_meta.csv', index=False)

merged = merged_all[merged_all['_merge'] == 'Matched rows'].drop(columns=['_merge', 'test_id_meta']).rename(columns={'test_id_data': 'test_id'})

merged['meta_file_date'] = pd.to_datetime(merged['meta_file_date'], format='%m/%d/%y')
merged['data_file_date'] = pd.to_datetime(merged['data_file_date'], format='%Y/%m/%d')

# Compare
merged.insert(4, 'date_match', merged['meta_file_date'] == merged['data_file_date'])
print(f"date matching: {sum(merged['date_match'])} / {len(merged)} ")

# Save the merged table
merged.to_csv('master_table.csv', index=False)


results of merging 2290 entries:
{'Matched rows': 2167, 'Unmatched rows in meta_table': 63, 'Unmatched rows in data_table': 60}
date matching: 2165 / 2167 


In [4]:
# merge sw table with master table
sw_table = pd.read_csv('sw_table.csv')
master_table = pd.read_csv('master_table.csv')

sw_table = sw_table.rename(columns={'test_id': 'test_id_sw'})

merged_all = pd.merge(
    master_table, sw_table,
    left_on='test_id', right_on='test_id_sw',
    how='outer',
    indicator=True
)

# Rename categorical labels properly
merged_all['_merge'] = merged_all['_merge'].cat.rename_categories({
    'left_only': 'Unmatched rows in master_table',
    'right_only': 'Unmatched rows in sw_table',
    'both': 'Matched rows'  # optional, keeps 'both' unchanged
})

# map merging result
counts = merged_all['_merge'].value_counts().to_dict()
print(f"results of merging {len(merged_all)} entries:")
print(counts)

# Reorder columns
first_cols = ['test_id_sw', 'swe_file_id']
sw_cols = [col for col in sw_table.columns if col not in first_cols]
other_cols = [col for col in merged_all.columns if col not in first_cols + sw_cols]

# Final column order
final_cols = first_cols + other_cols + sw_cols
merged_all = merged_all[final_cols]

# Keep records of unmatched rows
unmatched = merged_all[merged_all['_merge'] != 'Matched rows'].copy()

# Save
unmatched.to_csv('unmatched_records_sw_master.csv', index=False)

merged = merged_all[merged_all['_merge'] == 'Matched rows'].drop(columns=['_merge', 'test_id_sw'])

# Save the merged table
merged.to_csv('master_sw_table.csv', index=False)


results of merging 4162 entries:
{'Matched rows': 3779, 'Unmatched rows in master_table': 255, 'Unmatched rows in sw_table': 128}


In [3]:
df = pd.read_csv("master_sw_table.csv")

# Remove duplicates ignoring 'swe_file_id'
cols_without_id = df.columns.difference(['swe_file_id'])
df_unique = df.drop_duplicates(subset=cols_without_id)

print(f"Reduced from {len(df)} to {len(df_unique)} unique rows (ignoring 'swe_file_id')")

# Save the merged table
df_unique.to_csv('master_sw_table_no_duplicates.csv', index=False)

Reduced from 3779 to 1912 unique rows (ignoring 'sw_file_id')


In [60]:
# merge sw-mresults table with master table
sw_table = pd.read_csv('sw_mresults_table.csv')
master_table = pd.read_csv('master_sw_table_no_duplicates.csv')

sw_table.columns = ['SW_' + col for col in sw_table.columns]

sw_table = sw_table.rename(columns={'SW_test_id': 'test_id_sw', 'SW_swe_file_id': 'swm_file_id'}) 

merged_all = pd.merge(
    master_table, sw_table,
    left_on='test_id', right_on='test_id_sw',
    how='outer',
    indicator=True
)

# Rename categorical labels properly
merged_all['_merge'] = merged_all['_merge'].cat.rename_categories({
    'left_only': 'Unmatched rows in master_table',
    'right_only': 'Unmatched rows in sw_table',
    'both': 'Matched rows'  # optional, keeps 'both' unchanged
})

# map merging result
counts = merged_all['_merge'].value_counts().to_dict()
print(f"results of merging {len(merged_all)} entries:")
print(counts)

# Reorder columns
first_cols = ['test_id_sw', 'swm_file_id']
sw_cols = [col for col in sw_table.columns if col not in first_cols]
other_cols = [col for col in merged_all.columns if col not in first_cols + sw_cols]

# Final column order
final_cols = first_cols + other_cols + sw_cols
merged_all = merged_all[final_cols]

# Keep records of unmatched rows
unmatched = merged_all[merged_all['_merge'] != 'Matched rows'].copy()

# Save
unmatched.to_csv('unmatched_records_sw_master.csv', index=False)

merged = merged_all[merged_all['_merge'] == 'Matched rows'].drop(columns=['_merge', 'test_id_sw'])

# Save the merged table
merged.to_csv('master_sw_mresults_table.csv', index=False)


results of merging 1976 entries:
{'Matched rows': 1912, 'Unmatched rows in sw_table': 64, 'Unmatched rows in master_table': 0}
